# Interpretability in Deep Learning: A Hands-On Tour

---

## What is Interpretability?

Imagine you hire a brilliant consultant who gives you great advice — but can never explain *why*. You'd probably feel uneasy trusting them with high-stakes decisions. Modern machine learning models are exactly like that consultant: often accurate, rarely explainable.

**Interpretability** is the field that asks: *What did the model actually learn? Why did it make this prediction? Is it reasoning the right way, or just exploiting a lucky pattern?*

### Why Does It Matter?

- **Safety**: A self-driving car that works for the wrong reasons will fail in unexpected ways.
- **Fairness**: A loan model might achieve high accuracy by learning a protected attribute (race, gender) as a proxy.
- **Science**: In medicine or biology, we don't just want predictions — we want to *understand* the system.
- **Debugging**: When a model fails, interpretability tools help you figure out *why*.

### The Clever Hans Warning

Clever Hans was a horse in early 20th-century Germany, famous for apparently solving arithmetic problems. Investigators eventually discovered he wasn't calculating , he was reading subtle postural cues from his trainer. He had the right *outputs* for the wrong *reasons*.

Deep learning models do this constantly. **High accuracy ≠ correct reasoning.**

---

## What You'll Learn in This Notebook

| Section | Technique | What It Answers |
|---|---|---|
| 1 | Shortcut Learning | How models cheat |
| 2 | LIME | Why did THIS prediction happen? |
| 3 | SHAP | Which features matter globally? |
| 4 | Grad-CAM | Where did the CNN look? |

**Prerequisites**: Python, basic ML (train/test, sklearn), familiarity with neural networks at the conceptual level.

Let's dig in.

In [ ]:
#@title Install {display-mode: "form"}
# If you're on Colab or a fresh environment, uncomment the line below:
!pip install shap lime torch torchvision matplotlib seaborn scikit-learn numpy pandas captum --quiet

In [ ]:
#@title Import libraries {display-mode: "form"}
# ── Core scientific stack ──────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from matplotlib.colors import Normalize

# ── Sklearn ────────────────────────────────────────────────────────────────
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.inspection import permutation_importance
from sklearn.manifold import TSNE
from sklearn.neural_network import MLPClassifier

# ── LIME & SHAP ────────────────────────────────────────────────────────────
import lime
import lime.lime_tabular
import shap

# ── PyTorch ────────────────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# ── Reproducibility ────────────────────────────────────────────────────────
np.random.seed(42)
torch.manual_seed(42)

# ── Plot style ─────────────────────────────────────────────────────────────
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11
CMAP = 'RdBu_r'

print('All imports successful!')
print(f'PyTorch version: {torch.__version__}')

---
# Section 1 — Shortcut Learning: The Clever Hans Problem

## The Core Idea

Models learn to minimize loss on *training data*. If there's a feature in the training data that perfectly predicts the label but is meaningless in the real world, the model will happily use it. This is called **shortcut learning** (also called **Clever Hans behavior** or **spurious correlations**).

**Real-world examples:**
- A pneumonia classifier that learned to predict *lower* risk for asthma patients because asthma patients always went to the ICU and got aggressive treatment (so they had better outcomes in the training data).
- A cow vs. camel classifier that actually learned to detect *grass vs. sand* backgrounds.
- COVID X-ray classifiers that learned hospital-specific metadata (scanner type, patient position) instead of actual pathology.

## What We'll Do

We'll create a synthetic dataset with:
- 2 real signal features (genuinely predictive)
- 3 noise features (random, not predictive)
- 1 spurious feature (perfectly correlated with `y` in training, but random at test time)

Then we'll train a logistic regression and watch it get fooled.

> **💡 Key Insight**: A model that achieves 100% training accuracy isn't necessarily learning the right thing. Interpretability tools help us audit *what* was learned.


In [ ]:
np.random.seed(42)
N = 1000

# Real signal features: drawn from different distributions per class
y = np.random.randint(0, 2, size=N)           # binary labels
X_signal = np.column_stack([
    np.random.randn(N) + y * 1.5,             # feature 0: shifted by label
    np.random.randn(N) + y * 1.0,             # feature 1: smaller shift
])

# Noise features: completely random, independent of y
X_noise = np.random.randn(N, 3)               # features 2,3,4

# Spurious feature: nearly perfectly correlated with y (99% match in train)
# We'll flip it at test time to expose the shortcut
spurious_train = y.copy().astype(float) + np.random.randn(N) * 0.01
spurious_test  = np.random.randn(N)           # pure noise at test time

X_all_train = np.column_stack([X_signal, X_noise, spurious_train])
X_all_test  = np.column_stack([X_signal, X_noise, spurious_test])

# Train/test split (indices only, keeping our custom spurious feature)
idx = np.arange(N)
np.random.shuffle(idx)
train_idx, test_idx = idx[:700], idx[700:]

X_train = X_all_train[train_idx]
X_test  = X_all_test[test_idx]
y_train = y[train_idx]
y_test  = y[test_idx]

feature_names = ['signal_1', 'signal_2', 'noise_1', 'noise_2', 'noise_3', 'spurious']
print(f'Train size: {len(X_train)}, Test size: {len(X_test)}')
print(f'Features: {feature_names}')

In [ ]:
# ── Train logistic regression ──────────────────────────────────────────────
clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train, y_train)

train_acc = accuracy_score(y_train, clf.predict(X_train))
test_acc  = accuracy_score(y_test,  clf.predict(X_test))

print(f'Train accuracy: {train_acc:.3f}   <-- suspiciously high!')
print(f'Test  accuracy: {test_acc:.3f}    <-- dropped because spurious feature is now random')
print()

# ── Inspect coefficients ───────────────────────────────────────────────────
coefs = pd.Series(np.abs(clf.coef_[0]), index=feature_names).sort_values(ascending=False)
print('Coefficient magnitudes (absolute value):')
print(coefs.to_string())
print()
print('>>> The SPURIOUS feature has the largest coefficient!')
print('>>> The model learned the shortcut instead of the real signal.')

In [ ]:
# ── Permutation Importance: confirms the shortcut ──────────────────────────
# Permutation importance measures how much test accuracy DROPS when
# you randomly shuffle a feature (breaking its relationship with y).
# On test data, the spurious feature is already random — so shuffling it
# should matter very little. Let's check train vs test.

perm_train = permutation_importance(clf, X_train, y_train, n_repeats=30, random_state=42)
perm_test  = permutation_importance(clf, X_test,  y_test,  n_repeats=30, random_state=42)

In [ ]:
#@title Visualization Permutance importance {display-mode: "form"}
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, perm, split in zip(axes, [perm_train, perm_test], ['Train', 'Test']):
    means = perm.importances_mean
    stds  = perm.importances_std
    colors = ['#d73027' if n == 'spurious' else '#4575b4' for n in feature_names]
    bars = ax.barh(feature_names, means, xerr=stds, color=colors, alpha=0.85, capsize=4)
    ax.set_title(f'Permutation Importance ({split} set)', fontsize=13, fontweight='bold')
    ax.set_xlabel('Mean accuracy drop when feature is shuffled')
    ax.axvline(0, color='black', linewidth=0.8)

axes[0].set_title('Train: spurious feature looks VERY important', fontsize=12, fontweight='bold')
axes[1].set_title('Test: spurious feature is worthless (already noise)', fontsize=12, fontweight='bold')

fig.suptitle('Permutation Importance: Revealing the Shortcut', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nOn training data, the spurious feature looks crucial.')
print('On test data, it contributes nothing — exposing the shortcut.')

In [ ]:
#@title Logistic regression coefficient Magnitudes {display-mode: "form"}
# ── Coefficient bar chart ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#d73027' if n == 'spurious' else '#4575b4' for n in coefs.index]
ax.barh(coefs.index, coefs.values, color=colors, alpha=0.85)
ax.set_xlabel('|Coefficient|')
ax.set_title('Logistic Regression Coefficient Magnitudes', fontweight='bold')
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()

> **⚠️ Watch Out**: Permutation importance on *training* data can be misleading for overfitted models. Always verify on a held-out test set. The gap between train and test importance is itself a signal of a shortcut.

---

## 🏋️ Exercise

1. **Remove the spurious feature** entirely and retrain. What happens to train and test accuracy? Do the coefficients look healthier now?

2. **Weaken the shortcut**: Instead of 99% correlation, make the spurious feature only 80% correlated with `y` in training (hint: increase the noise: `spurious_train = y + np.random.randn(N) * X` — what value of X gives ~80%?). How does this affect the model's behavior?

3. **Bonus**: What if you had *no* signal features at all ,just noise and one spurious feature? What would the test accuracy be, and why?


In [ ]:
# 1)
# drop the last column
X_train_no_spurious = X_train[:, :5]
X_test_no_spurious  = X_test[:, :5]

mod_no_spur = LogisticRegression(max_iter=1000, random_state=42)
mod_no_spur.fit(X_train_no_spurious, y_train)

train_acc_ns = accuracy_score(y_train, mod_no_spur.predict(X_train_no_spurious))
test_acc_ns  = accuracy_score(y_test,  mod_no_spur.predict(X_test_no_spurious))

print(f'[with spurious]    Train: {train_acc:.3f}  Test: {test_acc:.3f}')
print(f'[without spurious]  Train: {train_acc_ns:.3f}  Test: {test_acc_ns:.3f}')

feature_names_ns = ['signal_1', 'signal_2', 'noise_1', 'noise_2', 'noise_3']
coefs_ns = pd.Series(np.abs(mod_no_spur.coef_[0]), index=feature_names_ns).sort_values(ascending=False)
print('\nCoefficient (without spurious):')
print(coefs_ns.to_string())

In [ ]:
#2)
np.random.seed(42)
spurious_train_weak = y.copy().astype(float) + np.random.randn(N) * 0.75

X_all_train_weak = np.column_stack([X_signal, X_noise, spurious_train_weak])
X_train_weak = X_all_train_weak[train_idx]

clf_weak = LogisticRegression(max_iter=1000, random_state=42)
clf_weak.fit(X_train_weak, y_train)

train_acc_w = accuracy_score(y_train, clf_weak.predict(X_train_weak))
test_acc_w  = accuracy_score(y_test,  clf_weak.predict(X_test))

print(f'[99% corr]  Train: {train_acc:.3f}  Test: {test_acc:.3f}')
print(f'[80% corr]  Train: {train_acc_w:.3f}  Test: {test_acc_w:.3f}')
print()
coefs_w = pd.Series(np.abs(clf_weak.coef_[0]), index=feature_names).sort_values(ascending=False)
print('Coefficient (80% correlation):')
print(coefs_w.to_string())

In [ ]:
#3
X_nosignal_train = np.column_stack([X_noise[train_idx], spurious_train[train_idx]])
X_nosignal_test  = np.column_stack([X_noise[test_idx],  spurious_test[test_idx]])

clf_nosig = LogisticRegression(max_iter=1000, random_state=42)
clf_nosig.fit(X_nosignal_train, y_train)

train_acc_bg = accuracy_score(y_train, clf_nosig.predict(X_nosignal_train))
test_acc_bg  = accuracy_score(y_test,  clf_nosig.predict(X_nosignal_test))

print(f'Train acc: {train_acc_bg:.3f}')
print(f'Test  acc: {test_acc_bg:.3f}')

---
# Section 2 — LIME: Local Explanations

## The Big Idea

Global model interpretation is hard — complex models carve out decision boundaries in ways that are difficult to summarize globally. But **locally**, near any single prediction, even a complex model can often be approximated by something simple.

**LIME** (Local Interpretable Model-agnostic Explanations, Ribeiro et al. 2016) exploits this:

1. Take a single prediction you want to explain.
2. Create many slightly perturbed versions of that input.
3. Run them through the black-box model to get predictions.
4. Fit a **simple linear model** (interpretable) on this synthetic neighborhood, weighted by distance to the original point.
5. The linear model's coefficients are your explanation.

**Analogy**: You can't describe the entire topography of a mountain range, but you can say "right where you're standing, the ground slopes slightly north-east."

> **💡 Key Insight**: LIME explanations are *local* — they explain a single prediction, not the model's overall behavior. Two nearby points can have very different LIME explanations if the decision boundary is complex in that region.

> **⚠️ Watch Out**: LIME explanations are **approximate**. The quality of the explanation depends on how well a linear model can approximate the local behavior. For highly non-linear boundaries, LIME can be misleading.


In [ ]:
# ── Load breast cancer dataset & train a black-box RF ─────────────────────
bc = load_breast_cancer()
X_bc, y_bc = bc.data, bc.target
feat_names_bc = bc.feature_names
class_names_bc = bc.target_names  # ['malignant', 'benign']

X_bc_train, X_bc_test, y_bc_train, y_bc_test = train_test_split(
    X_bc, y_bc, test_size=0.2, random_state=42, stratify=y_bc
)

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_bc_train, y_bc_train)

rf_test_acc = accuracy_score(y_bc_test, rf.predict(X_bc_test))
print(f'Random Forest test accuracy: {rf_test_acc:.4f}')
print(f'Dataset: {X_bc.shape[0]} samples, {X_bc.shape[1]} features')
print(f'Classes: {class_names_bc}')

In [ ]:
# ── Set up LIME explainer ─────────────────────────────────────────────────
explainer_lime = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_bc_train,
    feature_names=feat_names_bc,
    class_names=class_names_bc,
    mode='classification',
    random_state=42
)

# ── Explain 3 specific predictions ────────────────────────────────────────
# We'll pick: one confident correct benign, one confident correct malignant,
# and one that the model got wrong.

preds = rf.predict(X_bc_test)
probs = rf.predict_proba(X_bc_test)

# Find interesting samples
correct_benign   = np.where((preds == 1) & (y_bc_test == 1) & (probs[:,1] > 0.95))[0]
correct_malig    = np.where((preds == 0) & (y_bc_test == 0) & (probs[:,0] > 0.90))[0]
wrong_preds      = np.where(preds != y_bc_test)[0]

sample_ids = [correct_benign[0], correct_malig[0], wrong_preds[0] if len(wrong_preds) > 0 else correct_benign[1]]
sample_labels = ['Correct: Benign', 'Correct: Malignant', 'Incorrect prediction' if len(wrong_preds) > 0 else 'Correct: Benign (2)']

print(f'Selected samples: indices {sample_ids}')
for i, sid in enumerate(sample_ids):
    true_cls  = class_names_bc[y_bc_test[sid]]
    pred_cls  = class_names_bc[preds[sid]]
    conf      = probs[sid].max()
    print(f'  Sample {sid}: True={true_cls}, Pred={pred_cls}, Confidence={conf:.3f}  [{sample_labels[i]}]')

In [ ]:
#@title Visualization {display-mode: "form"}
# ── Generate and plot LIME explanations ───────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax_idx, (sid, label) in enumerate(zip(sample_ids, sample_labels)):
    # Generate LIME explanation — pass labels=(0,1) so both classes are stored
    exp = explainer_lime.explain_instance(
        data_row=X_bc_test[sid],
        predict_fn=rf.predict_proba,
        num_features=8,
        num_samples=1000,
        labels=(0, 1)          # fix: explicitly compute both class explanations
    )

    # Extract feature contributions for the predicted class
    pred_class = int(preds[sid])   # fix: cast np.int64 → int so dict lookup works
    explanation_list = exp.as_list(label=pred_class)
    feat_labels = [e[0] for e in explanation_list]
    feat_values = [e[1] for e in explanation_list]

    # Sort by absolute value
    sorted_pairs = sorted(zip(feat_labels, feat_values), key=lambda x: abs(x[1]))
    feat_labels_s = [p[0] for p in sorted_pairs]
    feat_values_s = [p[1] for p in sorted_pairs]

    colors = ['#d73027' if v < 0 else '#4575b4' for v in feat_values_s]
    ax = axes[ax_idx]
    ax.barh(range(len(feat_labels_s)), feat_values_s, color=colors, alpha=0.85)
    ax.set_yticks(range(len(feat_labels_s)))
    ax.set_yticklabels(feat_labels_s, fontsize=8)
    ax.axvline(0, color='black', linewidth=0.8)
    true_cls = class_names_bc[y_bc_test[sid]]
    pred_cls = class_names_bc[preds[sid]]
    ax.set_title(f'{label}\nTrue: {true_cls} | Pred: {pred_cls}', fontsize=10, fontweight='bold')
    ax.set_xlabel(f'Contribution to P({pred_cls})')

fig.suptitle('LIME Local Explanations — Breast Cancer Classifier', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print('Blue = pushes toward predicted class | Red = pushes against predicted class')

## 🏋️ Exercise

1. **Wrong prediction investigation**: Find the sample the model predicted incorrectly. Look at its LIME explanation. Do the features highlighted suggest why the model was confused? Does the local explanation make sense given what you know about breast cancer features (radius, texture, smoothness...)?

2. **Same-class comparison**: Find two samples that are both correctly classified as *benign*. Generate LIME explanations for both. Are the explanations similar (same features highlighted, same direction)? What does this tell you about how locally consistent the model is?

3. **Stability check**: Run LIME on the same sample three times with different `num_samples` (100, 500, 5000). How stable are the explanations? Does the ranking of features change?


In [ ]:
# ── Exercise 1 (LIME): Investigate the wrong prediction ───────────────────
misclassified = np.where(preds != y_bc_test)[0]
print(f'Misclassified: {len(misclassified)} samples -> indices: {misclassified}')

sid_wrong = misclassified[0]
true_cls_w = class_names_bc[y_bc_test[sid_wrong]]
pred_cls_w = class_names_bc[preds[sid_wrong]]
print(f'\nSample {sid_wrong}: True={true_cls_w}, Pred={pred_cls_w}, Conf={probs[sid_wrong].max():.3f}')

exp_wrong = explainer_lime.explain_instance(
    data_row=X_bc_test[sid_wrong],
    predict_fn=rf.predict_proba,
    num_features=10,
    num_samples=1000,
    labels=(0, 1)
)
pred_class_wrong = int(preds[sid_wrong])
explanation_wrong = exp_wrong.as_list(label=pred_class_wrong)

sorted_pairs = sorted(explanation_wrong, key=lambda x: abs(x[1]))
colors = ['#d73027' if v < 0 else '#4575b4' for _, v in sorted_pairs]

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh([p[0] for p in sorted_pairs], [p[1] for p in sorted_pairs], color=colors, alpha=0.85)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title(f'LIME — Wrong sample {sid_wrong}\nTrue: {true_cls_w} | Pred: {pred_cls_w}', fontweight='bold')
ax.set_xlabel(f'Contribution to P({pred_cls_w})')
plt.tight_layout()
plt.show()


In [ ]:
# ── Exercise 2 (LIME): Compare two correctly classified benign samples ─────
correct_benign_all = np.where((preds == 1) & (y_bc_test == 1))[0]
s1, s2 = correct_benign_all[0], correct_benign_all[1]
print(f'Comparing benign samples: {s1} and {s2}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, sid in zip(axes, [s1, s2]):
    exp = explainer_lime.explain_instance(
        data_row=X_bc_test[sid],
        predict_fn=rf.predict_proba,
        num_features=8,
        num_samples=1000,
        labels=(0, 1)
    )
    explanation = exp.as_list(label=1)
    sorted_pairs = sorted(explanation, key=lambda x: abs(x[1]))
    colors = ['#d73027' if v < 0 else '#4575b4' for _, v in sorted_pairs]
    ax.barh([p[0] for p in sorted_pairs], [p[1] for p in sorted_pairs], color=colors, alpha=0.85)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f'Sample {sid} (benign)\nConf={probs[sid, 1]:.3f}', fontweight='bold')
    ax.set_xlabel('Contribution to P(benign)')

fig.suptitle('LIME: comparing two benign samples', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print('If explanations differ a lot -> the decision boundary is non-linear in that region.')

In [ ]:
# ── Exercise 3 (LIME): Stability check with different num_samples ──────────
sid_stable = sample_ids[0]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, n_samples in zip(axes, [100, 500, 5000]):
    exp = explainer_lime.explain_instance(
        data_row=X_bc_test[sid_stable],
        predict_fn=rf.predict_proba,
        num_features=8,
        num_samples=n_samples,
        labels=(0, 1)
    )
    explanation = exp.as_list(label=int(preds[sid_stable]))
    sorted_pairs = sorted(explanation, key=lambda x: abs(x[1]))
    colors = ['#d73027' if v < 0 else '#4575b4' for _, v in sorted_pairs]
    ax.barh([p[0] for p in sorted_pairs], [p[1] for p in sorted_pairs], color=colors, alpha=0.85)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(f'num_samples={n_samples}', fontweight='bold')
    ax.set_xlabel('Contribution')

fig.suptitle(f'Stability Check — Sample {sid_stable}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print('With 100 samples explanations are noisy; with 5000 they converge to stable values.')

---
# Section 3 — SHAP: Principled Attribution

## From Local to Global

LIME explains individual predictions, but what if you want to understand the model's behavior *across the whole dataset*? SHAP (SHapley Additive exPlanations, Lundberg & Lee 2017) provides exactly this.

SHAP is grounded in **cooperative game theory**. The core idea:

> *How much does each feature contribute to pushing the prediction away from the average?*

This is measured using **Shapley values**, which fairly allocate credit among features by averaging over all possible orderings in which features could be introduced. They have nice mathematical properties:
- **Efficiency**: attributions sum to the difference between the prediction and the baseline
- **Symmetry**: equal features get equal credit
- **Null player**: features that don't affect the output get zero attribution

**For tree models** (Random Forests, XGBoost), SHAP can be computed exactly and efficiently using the TreeExplainer.

> **💡 Key Insight**: SHAP values decompose any prediction as:
> `prediction = base_value + SHAP(feature_1) + SHAP(feature_2) + ...`
> This additive decomposition is exact (not an approximation like LIME).

> **⚠️ Watch Out**: SHAP values tell you about feature contributions *within this model*. They don't tell you about causal importance in the real world — a feature can have high SHAP because it's correlated with another important feature.

## Reading the Plots

We'll generate three kinds of SHAP plots:

**Waterfall plot** (single prediction): Shows how each feature pushes the prediction up or down from the base value.

**Beeswarm plot** (whole dataset): Each dot is one sample. Color = feature value, x-position = SHAP value. Reveals how feature values relate to their impact.

**Bar plot** (mean |SHAP|): Global feature importance — average absolute SHAP value per feature.


In [ ]:
# ── SHAP with TreeExplainer ────────────────────────────────────────────────
# We reuse the Random Forest trained in Section 2

shap.initjs()  # initialize JavaScript for interactive plots (works in Jupyter)

# TreeExplainer computes exact SHAP values for tree-based models
explainer_shap = shap.TreeExplainer(rf)

# Compute SHAP values for the test set
# shap_values shape: (n_samples, n_features, n_classes) for classifiers
shap_values = explainer_shap(X_bc_test)

print(f'SHAP values shape: {shap_values.shape}')
print(f'Base value (average prediction): {explainer_shap.expected_value}')

# For binary classification, we'll use class 1 (benign) SHAP values
# shap_values[:, :, 1] selects class=1 contributions
shap_vals_class1 = shap_values[:, :, 1]

In [ ]:
# ── (a) Waterfall plot: explain one prediction ─────────────────────────────
# Pick the first test sample for demonstration
sample_idx = 0

plt.figure(figsize=(10, 6))
shap.plots.waterfall(shap_vals_class1[sample_idx], max_display=12, show=False)
plt.title(f'SHAP Waterfall Plot — Sample {sample_idx}\n'
          f'True: {class_names_bc[y_bc_test[sample_idx]]}, '
          f'Pred: {class_names_bc[rf.predict(X_bc_test[[sample_idx]])[0]]}',
          fontweight='bold', pad=15)
plt.tight_layout()
plt.show()
print('\nHow to read: Each bar shows how much that feature pushed the prediction')
print('up (blue) or down (red) from the base value (average model output).')
print(f'Base value = {explainer_shap.expected_value[1]:.3f}, Final prediction sum shown at top.')

In [ ]:
# ── (b) Beeswarm plot: global overview of all test samples ─────────────────
plt.figure(figsize=(10, 8))
shap.plots.beeswarm(shap_vals_class1, max_display=15, show=False)
plt.title('SHAP Beeswarm Plot — All Test Samples (Class: Benign)',
          fontweight='bold', pad=15)
plt.tight_layout()
plt.show()
print('\nHow to read:')
print('  • Each dot = one test sample')
print('  • X-position = SHAP value (impact on prediction)')
print('  • Color = feature value (red=high, blue=low)')
print('  • Features sorted by mean |SHAP| (most important at top)')
print('  • E.g., if high "worst radius" (red) → large negative SHAP → pushes toward malignant')

In [ ]:
# ── (c) Bar plot: mean absolute SHAP values ────────────────────────────────
plt.figure(figsize=(8, 7))
shap.plots.bar(shap_vals_class1, max_display=15, show=False)
plt.title('Mean |SHAP| — Global Feature Importance', fontweight='bold', pad=15)
plt.tight_layout()
plt.show()
print('\nHow to read: mean(|SHAP value|) over all test samples.')
print('This gives a simple ranking of which features the model relies on most.')

## 🏋️ Exercise

1. **Feature reduction**: Look at the SHAP bar plot. Identify the top 3 features by mean |SHAP|. Now train a new Random Forest using *only* those 3 features. How much accuracy do you lose compared to the full model? What does this tell you about feature redundancy in this dataset?

2. **SHAP interaction**: In the beeswarm plot, notice that some features have a mix of red and blue dots on both sides of zero. What does this mean? Can you find a feature where high values consistently push toward one class?

3. **LIME vs SHAP**: For the same misclassified sample you found in Section 2, compare the LIME explanation and the SHAP waterfall plot. Do they agree on which features are most important? Where do they disagree, and why might that be?

In [ ]:
# ── Exercise 1 (SHAP): Feature reduction with top-3 features ─────────────
mean_abs_shap = np.abs(shap_vals_class1.values).mean(axis=0)
top3_idx = np.argsort(mean_abs_shap)[-3:]
top3_names = [feat_names_bc[i] for i in top3_idx]
print('Top 3 features:', top3_names)

rf_top3 = RandomForestClassifier(n_estimators=100, random_state=42)
rf_top3.fit(X_bc_train[:, top3_idx], y_bc_train)
acc_top3 = accuracy_score(y_bc_test, rf_top3.predict(X_bc_test[:, top3_idx]))

print(f'Full model  (30 features): {rf_test_acc:.4f}')
print(f'Top-3 model  (3 features): {acc_top3:.4f}')
print(f'Accuracy drop: {rf_test_acc - acc_top3:.4f}')
print('\n-> With only 3 out of 30 features the drop is minimal: features are highly redundant.')

---
# Congratulations! You've completed the Interpretability Tour.

## What You've Learned

| Tool | What it Does | When to Use |
|---|---|---|
| **Permutation Importance** | Feature ranking by accuracy drop | Quick global importance on tabular data |
| **LIME** | Local linear approximation | Explaining individual predictions (any model) |
| **SHAP** | Shapley value attributions | Both local and global, principled decomposition |
| **Grad-CAM** | Spatial attention heatmap | CNNs: where did it look? |

## Key Takeaways

> **High accuracy does not mean the model learned the right thing.** Always audit what features or pixels it relies on.

> **Local and global explanations can disagree.** LIME explains a single point; SHAP aggregates over the dataset. Both are useful.

> **The choice of baseline matters for gradient-based methods.** There is no universally correct choice — it depends on what question you're asking.

> **Interpretability is not a silver bullet.** These tools all have failure modes and should be used as evidence, not verdicts.

## Further Reading

- **LIME**: Ribeiro et al. (2016) — "Why Should I Trust You?"
- **SHAP**: Lundberg & Lee (2017) — "A Unified Approach to Interpreting Model Predictions"
- **Grad-CAM**: Selvaraju et al. (2017) — "Grad-CAM: Visual Explanations from Deep Networks"

*Happy interpreting!*